In [0]:
"""Pharos CDM introspection, profiling, constraints, validation, and references."""


from __future__ import annotations


# --- Introspect ---

import enum as _enum
import typing
from dataclasses import dataclass

from pharos_cdm.meta import PharosMeta, iter_pharos_fields
from pharos_cdm.validation.registry import available_tables, resolve_model
from pydantic import BaseModel


@dataclass(frozen=True)
class FieldSpec:
    name: str
    meta: PharosMeta
    enum_cls: type | None
    optional: bool


@dataclass(frozen=True)
class TableSpec:
    name: str
    model: type[BaseModel]
    comment: str
    fields: tuple[FieldSpec, ...]
    pk: str
    pk_is_identity: bool
    parent: tuple[str, str] | None
    entity_fks: tuple[tuple[str, str, str], ...]
    ref_fks: tuple[tuple[str, str], ...]


def _unwrap_enum(annotation) -> tuple[type | None, bool]:
    optional = False
    args = typing.get_args(annotation)
    if args:
        non_none = [arg for arg in args if arg is not type(None)]
        optional = len(non_none) < len(args)
        annotation = non_none[0] if non_none else annotation
    if isinstance(annotation, type) and issubclass(annotation, _enum.Enum):
        return annotation, optional
    return None, optional


PARENT_OVERRIDES: dict[str, tuple[str, str]] = {
    "cohort": ("pharosid", "person"),
}


def _table_spec(name: str, model: type[BaseModel]) -> TableSpec:
    fields: list[FieldSpec] = []
    pk: str | None = None
    pk_ident = False
    parent: tuple[str, str] | None = None
    entity_fks: list[tuple[str, str, str]] = []
    ref_fks: list[tuple[str, str]] = []
    for fname, meta, finfo in iter_pharos_fields(model):
        enum_cls, optional = _unwrap_enum(finfo.annotation)
        fields.append(FieldSpec(fname, meta, enum_cls, optional))
        if meta.is_pk:
            pk, pk_ident = fname, meta.is_identity
        if meta.fk:
            ftable, fcol = meta.fk.split(".", 1)
            if ftable.startswith("ref_"):
                ref_fks.append((fname, meta.fk))
            else:
                entity_fks.append((fname, ftable, fcol))
                if parent is None:
                    parent = (fname, ftable)
    parent = PARENT_OVERRIDES.get(name, parent)
    if pk is None:
        raise ValueError(f"{name} has no declared primary key")
    return TableSpec(
        name=name,
        model=model,
        comment=getattr(model, "__table_comment__", ""),
        fields=tuple(fields),
        pk=pk,
        pk_is_identity=pk_ident,
        parent=parent,
        entity_fks=tuple(entity_fks),
        ref_fks=tuple(ref_fks),
    )


def build_registry() -> dict[str, TableSpec]:
    registry: dict[str, TableSpec] = {}
    for table in available_tables():
        _, model = resolve_model(table=table)
        registry[table] = _table_spec(table, model)
    roots = [name for name, spec in registry.items() if spec.parent is None]
    assert roots == ["person"], f"unexpected graph roots: {roots}"
    return registry


def topo_order(registry: dict[str, TableSpec]) -> list[str]:
    order: list[str] = []
    placed: set[str] = set()
    remaining = dict(registry)
    while remaining:
        progressed = False
        for table, spec in sorted(remaining.items()):
            dependencies = {target for _, target, _ in spec.entity_fks if target != table}
            if spec.parent:
                dependencies.add(spec.parent[1])
            if dependencies <= placed:
                order.append(table)
                placed.add(table)
                remaining.pop(table)
                progressed = True
                break
        if not progressed:
            raise ValueError(f"FK cycle among: {sorted(remaining)}")
    order.remove("person")
    order.insert(0, "person")
    return order


# --- Ids ---

import hashlib

import numpy as np

GROUP_TAG = {"breast": "BRE", "lung": "LUN", "pancreas": "PAN"}


def det_seed(base: int, key: str) -> int:
    digest = hashlib.sha256(f"{base}:{key}".encode()).hexdigest()
    return int(digest[:15], 16)


def det_rng(base: int, key: str) -> np.random.Generator:
    return np.random.default_rng(det_seed(base, key))


def mint_pharosids(group: str, n: int) -> list[str]:
    tag = GROUP_TAG[group]
    return [f"SYN-{tag}-{i:06d}" for i in range(1, n + 1)]


class IdentityMinter:
    def __init__(self, band_size: int = 1_000_000):
        self.band_size = band_size
        self._bands: dict[str, int] = {}
        self._next: dict[str, int] = {}

    def next(self, table: str) -> int:
        if table not in self._bands:
            self._bands[table] = (len(self._bands) + 1) * self.band_size
            self._next[table] = self._bands[table]
        self._next[table] += 1
        return self._next[table]


# --- Timeline ---

from dataclasses import dataclass
from datetime import date, timedelta

import numpy as np


@dataclass(frozen=True)
class Timeline:
    dob: date
    dx: date
    end: date
    death: date | None

    def between(self, rng: np.random.Generator, start: date, stop: date) -> date:
        span = max((stop - start).days, 0)
        return start + timedelta(days=int(rng.integers(0, span + 1)))

    def pre_dx(self, rng: np.random.Generator) -> date:
        return self.between(rng, self.dob + timedelta(days=365 * 18), self.dx)

    def post_dx(self, rng: np.random.Generator) -> date:
        return self.between(rng, self.dx, self.end)

    def offset_from_dx(self, days: int) -> date:
        value = self.dx + timedelta(days=days)
        lower = self.dob + timedelta(days=365 * 18)
        return max(lower, min(value, self.end))


def make_timeline(
    rng: np.random.Generator,
    today: date,
    death_rate: float = 0.18,
) -> Timeline:
    age_at_dx = int(rng.integers(28, 91))
    dx = today - timedelta(days=int(rng.integers(180, 365 * 8)))
    dob = dx - timedelta(days=age_at_dx * 365 + int(rng.integers(0, 365)))
    end = min(dx + timedelta(days=int(rng.integers(90, 365 * 6))), today)
    death = end if rng.random() < death_rate else None
    return Timeline(dob=dob, dx=dx, end=end, death=death)


def iso(value: date | None) -> str | None:
    return value.isoformat() if value else None


# --- Constraints ---

import numpy as np
from pharos_cdm.enums.person import EthnicGroup, Ethnicity, TumourGroup
from pharos_cdm.refs.ethnicity import parent_ethnicity_code
from pharos_cdm.refs.tnm import cancer_type_map


GROUP_CODE = {
    "breast": TumourGroup.BREAST.value,
    "pancreas": TumourGroup.PANCREATIC.value,
    "lung": TumourGroup.LUNG.value,
}
CODE2GROUP = {code: group for group, code in GROUP_CODE.items()}


def group_from_source(value) -> str | None:
    try:
        member = TumourGroup(value)
    except (ValueError, TypeError):
        return None
    return CODE2GROUP.get(member.value) if member else None


def sample_ethnicity_pair(rng: np.random.Generator) -> tuple[str, str | None]:
    ethnicities = list(Ethnicity)
    ethnicity = ethnicities[int(rng.integers(0, len(ethnicities)))]
    children = [
        group
        for group in EthnicGroup
        if parent_ethnicity_code(group.value) == ethnicity.value
    ]
    if not children or rng.random() < 0.3:
        return ethnicity.value, None
    child = children[int(rng.integers(0, len(children)))]
    return ethnicity.value, child.value


def sample_tnm(
    rng: np.random.Generator,
    enum_cls: type,
    group: str,
    profiled_codes: list[str] | None = None,
) -> str:
    applicability = cancer_type_map(enum_cls)
    valid = [member for member in enum_cls if group in applicability.get(member.name, set())]
    assert valid, f"no {enum_cls.__name__} members apply to {group}"
    if profiled_codes:
        allowed = set(profiled_codes)
        intersection = [member for member in valid if member.value in allowed]
        if intersection:
            valid = intersection
    return valid[int(rng.integers(0, len(valid)))].value


def applicable(field: FieldSpec, group: str) -> bool:
    return field.meta.tumour_groups is None or group in field.meta.tumour_groups


def null_inapplicable(row: dict, spec: TableSpec, group: str) -> dict:
    for field in spec.fields:
        if (
            not applicable(field, group)
            and field.optional
            and not field.meta.is_pk
            and field.name in row
        ):
            row[field.name] = None
    return row


def is_tnm_field(field: FieldSpec) -> bool:
    fk = field.meta.fk or ""
    return any(
        token in fk
        for token in ("_t_stage.", "_n_stage.", "_m_stage.", "_stage.")
    )


# --- Profile ---

import pandas as pd


QUANTILES = [i / 10 for i in range(11)]


def _ci_column(frame: pd.DataFrame, name: str) -> str | None:
    lookup = {str(column).lower(): str(column) for column in frame.columns}
    return lookup.get(name.lower())


def _canonicalize_columns(frame: pd.DataFrame, spec: TableSpec) -> pd.DataFrame:
    wanted = {field.name.lower(): field.name for field in spec.fields}
    renames = {
        column: wanted[str(column).lower()]
        for column in frame.columns
        if str(column).lower() in wanted
    }
    return frame.rename(columns=renames)


def _prepare_frame(frame: pd.DataFrame, spec: TableSpec) -> pd.DataFrame:
    """Canonicalize CDM columns and recover the source person key.

    The current silver output carries a populated ``person_id`` extra while
    ``pharosid`` is present but null. Profiling may use that source-only key for
    joins; generated identifiers remain entirely constructive.
    """
    prepared = _canonicalize_columns(frame.copy(), spec)
    person_id = _ci_column(prepared, "person_id")
    if "pharosid" not in prepared.columns:
        if person_id is not None:
            prepared["pharosid"] = prepared[person_id]
        return prepared
    if person_id is None:
        return prepared
    usable = prepared["pharosid"].notna() & prepared["pharosid"].astype(str).str.strip().ne("")
    prepared["pharosid"] = prepared["pharosid"].where(usable, prepared[person_id])
    return prepared


def _parse_enum_member(enum_cls, value):
    """Parse bare codes/labels plus silver's common ``code - label`` form."""
    if pd.isna(value):
        return None
    candidates = [value]
    if isinstance(value, str) and " - " in value:
        code, label = value.strip().split(" - ", 1)
        candidates.extend((code.strip(), f"{code.strip()} {label.strip()}"))
    for candidate in candidates:
        try:
            member = enum_cls(candidate)
        except (ValueError, TypeError):
            continue
        if member is not None:
            return member
    return None


def _group_lookup(frames: dict[str, pd.DataFrame]) -> pd.Series | None:
    person = frames.get("person")
    if person is None:
        return None
    pid = _ci_column(person, "pharosid")
    group = _ci_column(person, "tumour_group")
    if pid is None or group is None:
        return None
    normalized = pd.DataFrame(
        {
            "pharosid": person[pid],
            "_grp": person[group].map(group_from_source),
        }
    ).dropna(subset=["pharosid", "_grp"])
    if normalized.empty:
        return None

    def modal_group(values: pd.Series) -> str:
        counts = values.value_counts()
        winners = sorted(counts[counts == counts.max()].index)
        return winners[0]

    return normalized.groupby("pharosid", sort=False)["_grp"].agg(modal_group)


def _dx_anchor(frames: dict[str, pd.DataFrame]) -> pd.Series | None:
    tumour = frames.get("tumour")
    if tumour is None:
        return None
    pid = _ci_column(tumour, "pharosid")
    dx_col = _ci_column(tumour, "date_of_diagnosis")
    if pid is None or dx_col is None:
        return None
    dx = pd.to_datetime(tumour[dx_col], errors="coerce")
    return dx.groupby(tumour[pid]).min()


def _categorical(sub: pd.DataFrame, column: str, enum_cls, min_support: int):
    values = sub[[column, "pharosid"]].dropna(subset=[column])
    parsed: list[str | None] = []
    invalid = 0
    for value in values[column]:
        member = _parse_enum_member(enum_cls, value)
        if member is None:
            invalid += 1
            parsed.append(None)
        else:
            parsed.append(member.value)
    values = values.assign(_code=parsed).dropna(subset=["_code"])
    support = values.groupby("_code")["pharosid"].nunique()
    keep = support[support >= min_support]
    dropped_low = int((support < min_support).sum())
    if keep.empty:
        return None, invalid, dropped_low
    counts = values[values["_code"].isin(keep.index)]["_code"].value_counts()
    weights = counts / counts.sum()
    return (
        {"codes": list(weights.index), "weights": [float(value) for value in weights]},
        invalid,
        dropped_low,
    )


def _numeric(sub: pd.DataFrame, column: str):
    values = pd.to_numeric(sub[column], errors="coerce").dropna()
    if len(values) < 5:
        return None
    return {"quantiles": [float(value) for value in values.quantile(QUANTILES)]}


def _date_offset(sub: pd.DataFrame, column: str, anchors: pd.Series):
    dates = pd.to_datetime(sub[column], errors="coerce")
    anchor_dates = sub["pharosid"].map(anchors)
    offsets = (dates - anchor_dates).dt.days.dropna()
    if len(offsets) < 5:
        return None
    return {
        "offset_quantiles_days": [int(value) for value in offsets.quantile(QUANTILES)]
    }


def build_profile(
    frames: dict[str, pd.DataFrame],
    registry: dict[str, TableSpec],
    min_support: int = 10,
    source: str = "",
) -> dict:
    prepared_frames = {
        table: _prepare_frame(frame, registry[table])
        for table, frame in frames.items()
        if table in registry and frame is not None
    }
    groups = _group_lookup(prepared_frames)
    anchors = _dx_anchor(prepared_frames)
    profile = {"source": source, "min_support": min_support, "tables": {}}
    if groups is None:
        return profile

    for table, spec in registry.items():
        source_frame = prepared_frames.get(table)
        if source_frame is None or source_frame.empty:
            continue
        frame = source_frame.copy()
        if "pharosid" not in frame.columns:
            continue
        frame = frame.assign(_grp=frame["pharosid"].map(groups))
        table_profile = {"n_rows": int(len(frame)), "seqlen": {}, "columns": {}}

        if spec.parent:
            fk_col, parent = spec.parent
            parent_source = prepared_frames.get(parent)
            if parent_source is not None:
                parent_frame = parent_source.copy()
                parent_pk = registry[parent].pk
                if (
                    fk_col in frame.columns
                    and parent_pk in parent_frame.columns
                    and "pharosid" in parent_frame.columns
                ):
                    columns = list(dict.fromkeys([parent_pk, "pharosid"]))
                    parents = (
                        parent_frame[columns]
                        .dropna(subset=[parent_pk])
                        .drop_duplicates(parent_pk)
                    )
                    parents = parents.assign(_grp=parents["pharosid"].map(groups))
                    counts = frame.groupby(fk_col).size()
                    parents = parents.assign(
                        _n=parents[parent_pk].map(counts).fillna(0).astype(int)
                    )
                    for group, group_parents in parents.dropna(subset=["_grp"]).groupby("_grp"):
                        histogram = group_parents["_n"].value_counts().sort_index()
                        table_profile["seqlen"][group] = {
                            "counts_hist": {
                                str(key): int(value) for key, value in histogram.items()
                            }
                        }

        fields = {field.name: field for field in spec.fields}
        for column, field in fields.items():
            if column not in frame.columns or field.meta.is_pk or field.meta.fk:
                continue
            entry = {
                "kind": None,
                "null_rate": float(frame[column].isna().mean()),
                "by_group": {},
                "dropped_invalid": 0,
                "dropped_low_support": 0,
            }
            for group, group_frame in frame.dropna(subset=["_grp"]).groupby("_grp"):
                if field.enum_cls is not None:
                    found, invalid, low = _categorical(
                        group_frame, column, field.enum_cls, min_support
                    )
                    entry["kind"] = "categorical"
                    entry["dropped_invalid"] += invalid
                    entry["dropped_low_support"] += low
                elif (field.meta.sql_type or "").upper() in ("DATE", "TIMESTAMP"):
                    found = (
                        _date_offset(group_frame, column, anchors)
                        if anchors is not None
                        else None
                    )
                    entry["kind"] = "date_offset"
                elif (field.meta.sql_type or "").upper() in (
                    "BIGINT",
                    "INT",
                    "SMALLINT",
                    "DOUBLE",
                    "FLOAT",
                    "DECIMAL",
                ):
                    found = _numeric(group_frame, column)
                    entry["kind"] = "numeric"
                else:
                    found = None
                if found:
                    entry["by_group"][group] = found
            if entry["by_group"]:
                table_profile["columns"][column] = entry
        profile["tables"][table] = table_profile
    return profile


# --- Values ---

import numpy as np



def may_null(field: FieldSpec, group: str) -> bool:
    # Some group-scoped CDM fields remain non-Optional in the Pydantic model.
    # They still need a typed value outside their metadata scope so model
    # validation can load the row; only Optional annotations may become null.
    return field.optional and not (field.meta.mandatory and applicable(field, group))


FIELD_RANGES: dict[tuple[str, str], tuple[float, float]] = {
    ("tumour", "invasive_size_clinical"): (1.0, 120.0),
    ("tumour", "total_size_clinical"): (1.0, 150.0),
}
_DEFAULT_INT = (0, 6)
_DEFAULT_DBL = (0.0, 10.0)


def _prof_entry(profile, table: str, field: str, group: str):
    try:
        entry = profile["tables"][table]["columns"][field]
        return entry, entry["by_group"][group]
    except (KeyError, TypeError):
        return None, None


def _interp_quantiles(rng: np.random.Generator, quantiles: list[float]) -> float:
    index = int(rng.integers(0, len(quantiles) - 1))
    return float(quantiles[index]) + float(rng.random()) * (
        float(quantiles[index + 1]) - float(quantiles[index])
    )


def _spec_date(name: str, rng: np.random.Generator, timeline: Timeline) -> str | None:
    lower = name.lower()
    if "dob" in lower or "birth" in lower:
        return iso(timeline.dob)
    if "death" in lower:
        return iso(timeline.death)
    if "diag" in lower or lower.endswith("_dx") or lower.startswith("dx"):
        return iso(timeline.dx)
    return iso(timeline.post_dx(rng))


def sample_field(
    rng: np.random.Generator,
    table: str,
    field: FieldSpec,
    timeline: Timeline,
    group: str,
    prof: dict | None = None,
):
    sql_type = (field.meta.sql_type or "STRING").upper()
    entry, pool = _prof_entry(prof, table, field.name, group)

    if pool is not None:
        if may_null(field, group) and rng.random() < entry.get("null_rate", 0.0):
            return None
        if entry["kind"] == "categorical":
            index = int(rng.choice(len(pool["codes"]), p=pool["weights"]))
            return pool["codes"][index]
        if entry["kind"] == "numeric":
            value = _interp_quantiles(rng, pool["quantiles"])
            if sql_type in ("BIGINT", "INT", "SMALLINT"):
                return int(round(value))
            return round(value, 2)
        if entry["kind"] == "date_offset":
            days = int(round(_interp_quantiles(rng, pool["offset_quantiles_days"])))
            value = timeline.offset_from_dx(days)
            return value.isoformat() + ("T00:00:00" if sql_type == "TIMESTAMP" else "")

    if field.enum_cls is not None:
        members = list(field.enum_cls)
        return members[int(rng.integers(0, len(members)))].value
    if sql_type == "DATE":
        return _spec_date(field.name, rng, timeline)
    if sql_type == "TIMESTAMP":
        return timeline.post_dx(rng).isoformat() + "T00:00:00"
    if sql_type in ("BIGINT", "INT", "SMALLINT"):
        low, high = FIELD_RANGES.get((table, field.name), _DEFAULT_INT)
        return int(rng.integers(int(low), int(high) + 1))
    if sql_type in ("DOUBLE", "FLOAT", "DECIMAL"):
        low, high = FIELD_RANGES.get((table, field.name), _DEFAULT_DBL)
        return round(float(rng.uniform(low, high)), 2)
    if sql_type == "BOOLEAN":
        return bool(rng.random() < 0.5)
    return f"synthetic {table}.{field.name} #{int(rng.integers(1, 10_000))}"


# --- Refs Tables ---

import pandas as pd
from pharos_cdm.enums.person import EthnicGroup, Ethnicity
from pharos_cdm.enums.tnm import (
    ClinicalMStage,
    ClinicalNStage,
    ClinicalStage,
    ClinicalTStage,
    PathologicalMStage,
    PathologicalNStage,
    PathologicalStage,
    PathologicalTStage,
)
from pharos_cdm.refs import tnm as tnm_ref
from pharos_cdm.refs.ethnicity import parent_ethnicity_code

_TNM = {
    "ref_clinical_t_stage": ClinicalTStage,
    "ref_clinical_n_stage": ClinicalNStage,
    "ref_clinical_m_stage": ClinicalMStage,
    "ref_clinical_stage": ClinicalStage,
    "ref_pathological_t_stage": PathologicalTStage,
    "ref_pathological_n_stage": PathologicalNStage,
    "ref_pathological_m_stage": PathologicalMStage,
    "ref_pathological_stage": PathologicalStage,
}


def _scalarize(value):
    if hasattr(value, "value") and hasattr(value, "label"):
        return value.value
    if isinstance(value, (list, tuple)):
        return "; ".join(str(item) for item in value)
    if isinstance(value, (set, frozenset)):
        return "; ".join(sorted(str(item) for item in value))
    return value


def _rows_from_module(module) -> pd.DataFrame:
    from pydantic import BaseModel

    for value in vars(module).values():
        if isinstance(value, (list, tuple)) and value and isinstance(value[0], BaseModel):
            return pd.DataFrame(
                [
                    {key: _scalarize(item) for key, item in row.model_dump().items()}
                    for row in value
                ]
            )
    raise ValueError(f"no row container found in {module.__name__}")


def build_ref_tables() -> dict[str, pd.DataFrame]:
    refs: dict[str, pd.DataFrame] = {}
    refs["ref_ethnicity"] = pd.DataFrame(
        [
            {
                "ethnic_group_code": group.value,
                "ethnic_group_label": group.label,
                "ethnicity_code": parent_ethnicity_code(group.value),
                "ethnicity_label": Ethnicity(parent_ethnicity_code(group.value)).label,
            }
            for group in EthnicGroup
        ]
    )
    for name, enum_cls in _TNM.items():
        rows = tnm_ref._build_scale_reference(enum_cls)
        refs[name] = pd.DataFrame(
            [
                {
                    "code": row.code,
                    "label": row.label,
                    "cancer_types": ",".join(sorted(row.cancer_types)),
                }
                for row in rows
            ]
        )
    from pharos_cdm.refs import cancer_type, treatment

    refs["ref_cancer_type"] = _rows_from_module(cancer_type)
    refs["ref_treatment"] = _rows_from_module(treatment)
    return refs


# --- Validate ---

import tempfile
from pathlib import Path

from pharos_cdm.validation.cross_table.followup import check_followup_consistency
from pharos_cdm.validation.cross_table.medical_history import check_family_history_alignment
from pharos_cdm.validation.cross_table.pathology import check_pathology_microinvasion
from pharos_cdm.validation.runner import validate_table


GROUPS = ("breast", "lung", "pancreas")


def validate_dataset(dataset) -> dict:
    group_of = {
        pid: CODE2GROUP.get(code)
        for pid, code in zip(
            dataset.tables["person"]["pharosid"],
            dataset.tables["person"]["tumour_group"],
        )
    }
    report = {"by_table": {}, "issues": []}
    with tempfile.TemporaryDirectory() as temp_dir:
        for table, frame in dataset.tables.items():
            error_count = 0
            for group in GROUPS:
                subset = (
                    frame[frame["pharosid"].map(group_of) == group]
                    if "pharosid" in frame.columns
                    else frame
                )
                if subset.empty:
                    continue
                path = Path(temp_dir) / f"{table}_{group}.parquet"
                subset.to_parquet(path, index=False)
                table_report = validate_table(path, table=table, tumour_group=group)
                errors = [issue for issue in table_report.issues if issue.severity == "error"]
                error_count += len(errors)
                report["issues"].extend(
                    {
                        "table": table,
                        "group": group,
                        "row": issue.row,
                        "field": issue.field,
                        "message": issue.message,
                    }
                    for issue in errors[:20]
                )
            report["by_table"][table] = error_count

    selected = {
        table: dataset.tables[table].where(pd.notna(dataset.tables[table]), None).to_dict("records")
        for table in (
            "followup",
            "local_recurrence",
            "distant_metastasis",
            "medical_history",
            "family_cancer_history",
            "pathology",
            "pathology_focus",
        )
    }
    cross_table = (
        check_followup_consistency(
            selected["followup"],
            local_recurrence_rows=selected["local_recurrence"],
            distant_metastasis_rows=selected["distant_metastasis"],
        )
        + check_family_history_alignment(
            selected["medical_history"], selected["family_cancer_history"]
        )
        + check_pathology_microinvasion(
            selected["pathology"], selected["pathology_focus"]
        )
    )
    cross_errors = [issue for issue in cross_table if issue.severity == "error"]
    report["cross_table_errors"] = len(cross_errors)
    report["issues"].extend(
        {"table": "cross_table", "message": issue.message}
        for issue in cross_errors[:40]
    )
    report["errors"] = sum(report["by_table"].values()) + report["cross_table_errors"]
    return report


# Imported late solely to keep the public module imports compact.
import pandas as pd


# --- Sources ---

SOURCE_TABLE_MAP = {
    "person": "pharos_person",
    "cohort": "pharos_cohort",
    "person_tumour_group": "pharos_tumour_group",
    "medical_history": "pharos_medical_history",
    "family_cancer_history": "pharos_family_cancer_history",
    "personal_cancer_history": "pharos_personal_history",
    "comorbidity": "pharos_comorbidities",
    "medication": "pharos_medication",
    "imaging": "pharos_imaging",
    "tumour": "pharos_tumour",
    "pathology": "pharos_pathology",
    "pathology_focus": "pharos_pathology_tumourfocus",
    "sample": "pharos_sample",
    "treatment": "pharos_treatment",
    "followup": "pharos_followup",
    "local_recurrence": "pharos_recurrence",
    "distant_metastasis": "pharos_metastasis",
}